# Borehole descriptions as a sequence-labelling problem

This notebook introduces the BOVINO data before fitting or comparing models. It follows three questions:

1. What does one training example represent?
2. Which information is available to the model?
3. Why can an apparently accurate spatial model still be unreliable?

The figures and counts come from the real study data. No model is trained here.

## 1. Boreholes, campaigns, and geological descriptions

The dataset contains **95 boreholes**, **1,100 unique descriptions**, and **11 survey campaigns** acquired between 1989 and 2023. Each borehole becomes an ordered sequence of depth intervals. The prediction target is one of six geotechnical units for every valid interval.

![Borehole locations grouped by survey campaign](../assets/survey_campaign_map.png)

Different campaigns also mean different operators, writing styles, terminology, and spatial coverage. This makes the campaign identifier scientifically relevant even though it is not used as a predictive feature.

## 2. One borehole is a vertical sequence

![Example of a source borehole log](../assets/borehole_log_example.png)

Descriptions provide direct evidence about material type, consistency, inclusions, alteration, and fracturing. Adjacent intervals also constrain one another: geological units tend to persist with depth and transitions occur in sequence. The task is therefore sequence labelling rather than independent sentence classification.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import bovino_results as br

predictions = br.load_predictions('lobo_predictions.csv')
summary = {
    'boreholes': predictions['borehole'].nunique(),
    'evaluated intervals': len(predictions),
    'maximum depth (m)': predictions['depth'].max(),
    'classes': predictions['true_class_id'].nunique(),
}
summary

## 3. What the model receives

The controlled experiments separate three sources of information:

- **Depth**, which describes vertical position within a borehole.
- **XYZ coordinates**, which encode strong local spatial structure.
- **Text embeddings**, which compress the geological descriptions into numerical vectors.

The principal text representation is MiniLM reduced from 384 dimensions to 16 principal components. PCA is fitted inside each training fold before transforming the held-out group.

## 4. What the model does not receive

The reference labels reflect more than the exported table. A geotechnical expert can also use pocket-penetrometer measurements, piezometric information, and prior knowledge of the area. These sources are not available to the model.

This missing information sets a real ceiling on the task. Uncertainty cannot recover evidence that was never supplied, but it should warn us when the available evidence no longer supports a reliable prediction.

## Take-home message

The model combines a vertically ordered text record with a strong spatial prior. The validation protocol must therefore test both transfer to new boreholes and transfer beyond familiar campaigns or nearby spatial support.